# qchem_colab_app UMA版 — 機械学習ポテンシャルによる構造最適化・振動数解析

Google Colab上で、機械学習ポテンシャル **UMA**(Meta FAIR Chemistry)を使った
構造最適化と振動数解析を行います。DFT等の量子化学計算よりはるかに高速ですが、
**結果は機械学習による予測値であり、量子化学計算そのものの値ではありません。**
(量子化学計算を行う版は、同じリポジトリの `ColabApp.ipynb` です。)

**上から順にセルを実行**していく構成になっています。

**初回の準備(下の「事前準備」: Hugging Faceのトークン)が済んでいれば、メニューの「ランタイム」→**
**「すべてのセルを実行」を押すだけで、最後に操作画面が表示されるまで自動で進みます**
(最初に1回だけ、トークンの入力が必要です。Colabの「シークレット」に保存しておけば、その入力も不要です)。

- **I. Setup Section** — 環境構築(ランタイム設定・コードとライブラリの準備・Hugging Faceの認証)
- **II. Execution Section** — 実際の計算画面(GUI)

初めてJupyter/Colabを使う方へ: 各コードセルの左側にマウスを合わせると再生ボタン(▶)が
表示されます。それをクリックするか、セルを選択して `Shift+Enter` を押すと、そのセルが実行されます。
**必ず上のセルから順番に**実行してください(途中を飛ばすとエラーになります)。

### ⚠ UMAの利用条件について(必ずお読みください)

- UMAの学習済みモデルは **FAIR Chemistry License v1** の下で提供されており、
  Hugging Face( https://huggingface.co/facebook/UMA )で利用条件に同意した方だけが使えます。
  同意は、法的な同意能力のある方が行ってください(未成年の方は、保護者や指導者の方と相談してください)。
- 利用にあたっては、上記ページに掲載されている **Acceptable Use Policy**(軍事・戦争・原子力関連の用途、
  武器の開発、違法薬物、他者を欺く行為などへの利用の禁止)を守ってください。
- 結果を論文・報告書などで公表する場合は、UMAを使用したことを明記してください。
  引用: Wood, B. M. et al. *UMA: A Family of Universal Models for Atoms.* arXiv:2506.23971 (2025).

## 事前準備: Hugging Faceの利用申請とトークンの入力(初回のみ)

UMAの学習済みモデルは、利用条件に同意した方だけがダウンロードできます。
**この手順は、法的な同意能力のある方(教員・保護者・指導者など)が行ってください。**
未成年の方が使う場合は、そのような方があらかじめ設定したセッションで利用してください。

1. https://huggingface.co でアカウントを作成してログインします。
2. https://huggingface.co/facebook/UMA を開き、**FAIR Chemistry License v1 と Acceptable Use Policy を読んだうえで**、
   フォームに入力して利用条件に同意し、アクセスを申請します(承認まで時間がかかる場合があります)。
3. https://huggingface.co/settings/tokens で、権限に **Read** を含む **アクセストークン**(`hf_` で始まる文字列)を作成します。
4. (おすすめ)Colabの左側の「シークレット」(カギのアイコン)に、名前を `HF_TOKEN`、値をトークンとして保存し、
   「ノートブックからのアクセス」をオンにします。以後は、入力なしで自動的に使われます。

### このノートブックの使い方

メニューの **「ランタイム」→「すべてのセルを実行」** を押します。

- シークレットに保存していない場合は、**すぐ下のセルに入力欄が表示されます。** トークンを貼り付けて
  Enterキーを押してください(入力内容は画面に表示されません)。**入力が必要なのは、この最初の1回だけです。**
- 以降は自動で進みます(初回は、ライブラリのインストールに数分かかります)。
- 最後に操作画面が表示されます。
- 途中で警告や「セッションを再起動」の案内が表示された場合は、その指示に従ってください。

**トークンは、下のセルの入力欄か、Colabのシークレットにだけ入力してください。**
他人に教えたり、ノートブックやGitHubに書き込んだり、コードに直接書いたりしないでください
(このノートブックでは、トークンを画面・ログ・保存されるファイルに残さないようにしています)。

In [ ]:
#@title 事前準備. Hugging Faceのアクセストークンを入力する(最初に1回だけ入力が必要です)
# トークンを入れ直したい場合は、右のチェックを入れてから実行してください。
FORCE_REENTER = False  #@param {type:"boolean"}

import os

_token = None
if os.environ.get("HF_TOKEN") and not FORCE_REENTER:
    print("✓ Hugging Faceのトークンは、このセッションで設定済みです。入力は不要です。")
else:
    try:
        from google.colab import userdata
        _token = userdata.get("HF_TOKEN")     # Colabのシークレットに保存してある場合
    except Exception:
        _token = None                         # シークレット未設定・未許可の場合は、下の入力欄を使う

    if _token:
        print("Colabのシークレット(HF_TOKEN)を使用します。")
    else:
        from getpass import getpass
        _token = getpass("Hugging Faceのアクセストークン(hf_ で始まる文字列)を入力して、"
                         "Enterキーを押してください(入力内容は表示されません): ").strip()

    if _token:
        # 受け渡しは、Hugging Face公式の標準の環境変数 HF_TOKEN だけで行う
        # (ログインとアクセス確認は、後のセルで uma_app が行う)。
        os.environ["HF_TOKEN"] = _token
        print("✓ トークンを受け付けました。以降のセルは自動で進みます。")
    else:
        print("⚠ トークンが入力されませんでした。UMAを使うには、このセルをもう一度実行してください。")
_token = None                                 # 変数にトークンを残さない

## I. Setup Section

### 0. ランタイムの設定(通常は、ファイルを開いたときに自動で反映されます)

上部メニューの **「ランタイム」→「ランタイムのタイプを変更」** を開き、次のとおり設定してください。

| 項目 | 設定 | 理由 |
|---|---|---|
| **ハードウェア アクセラレータ** | **T4 GPU** | UMAはGPUで高速に動作します(このノートブックには既定でT4が指定されています) |
| **ランタイム バージョン** | **2026.07** | Python 3.12 / PyTorch 2.11.0 などを固定し、Google側の更新で動作が変わるのを防ぎます |

このノートブックには、**ランタイム バージョン 2026.07 と T4 GPU が設定として記録されています**
(Colabがファイルに保存する `runtime_attributes` / `gpuType` です)。ファイルを開いたときに反映されない場合は、
上の表のとおり手動で設定してください。下のセルで、現在の環境を確認できます
(違っていても動作は続行しますが、動作確認は上記の環境で行っています)。

過去のランタイムバージョンは、接続に2〜3分かかります(公式情報:
https://research.google.com/colaboratory/runtime-version-faq.html )。

In [ ]:
#@title 0. 現在の実行環境を確認する
import importlib.metadata as md
import platform
import subprocess

# torchは、「2. 依存ライブラリのインストール」で fairchem-core により 2.11 → 2.13 に更新される
# ことを確認しているため、どちらの版も想定内として扱う。
EXPECTED = {"python": ("3.12",), "numpy": ("2.0",), "torch": ("2.11", "2.13")}

def _version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

print(f"Python : {platform.python_version()}")
print(f"numpy  : {_version('numpy')}")
print(f"torch  : {_version('torch')}")

try:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=15).stdout.strip()
except (FileNotFoundError, OSError, subprocess.TimeoutExpired):
    gpu_name = ""
print(f"GPU    : {gpu_name or '(検出されませんでした)'}")

problems = []
if not platform.python_version().startswith(EXPECTED["python"]):
    problems.append(f"Pythonが {EXPECTED['python'][0]}.x ではありません(ランタイムバージョンが 2026.07 でない可能性があります)")
for pkg in ("numpy", "torch"):
    v = _version(pkg)
    if v is None or not v.startswith(EXPECTED[pkg]):
        problems.append(f"{pkg} が {' または '.join(EXPECTED[pkg])}.x ではありません")
if "T4" not in gpu_name:
    problems.append("T4 GPUが検出されませんでした(GPUなしではUMAの計算に非常に時間がかかります)")

if problems:
    print("\n⚠ 次の点が想定と異なります:")
    for p in problems:
        print("  -", p)
    print("「ランタイム」→「ランタイムのタイプを変更」で設定を確認してください。"
          "設定を変えるとセッションが再接続されるため、このセルから実行し直してください。")
else:
    print("\n✓ 想定どおりの環境です(ランタイム 2026.07 / T4 GPU)。")
print("(torchは、「2.」のインストール後に 2.11.0 → 2.13.0 に更新されます)")

# 以降のインストールで torch / numpy が入れ替わったかを調べるための記録
VERSIONS_BEFORE_INSTALL = {"numpy": _version("numpy"), "torch": _version("torch"),
                           "ipywidgets": _version("ipywidgets")}

### 1. コード本体の取得(GitHubから・ブランチを選んで取得)

このプロジェクトのコードは https://github.com/orgchem610/qchem_colab_app で管理しています。
**どのブランチ(またはバージョン)のコードを使うかは、「1-b」のセルの `BRANCH` 欄で選びます。**

1. (任意)「1-a」のセルを実行すると、GitHub上にあるブランチとタグ(バージョン)の一覧が表示されます。
2. 「1-b」のセルの `BRANCH` 欄で、プルダウンから選ぶか、一覧にない名前を直接入力します。
   - 通常のご利用: `main`(安定版)
   - 開発中の版の動作確認: `develop`、または `feature/...` などのブランチ名
   - 過去のバージョンの再現: `v0.2.0` などのタグ名
3. 「1-b」のセルを実行します。まだ取得していなければ `git clone` を、取得済みなら選んだブランチ(タグ)へ
   切り替えて最新版を取得します。

**⚠ すでにGUIを表示した後に、別のブランチへ切り替えた場合**は、
「ランタイム」→「セッションを再起動」を実行してから、「1-b」のセルを含めて上から順に実行し直してください
(Pythonが古いコードを読み込んだままになるためです。取得済みのコードや、インストール済みのライブラリは
セッションを再起動しても残ります)。

In [ ]:
#@title 1-a. (任意) GitHub上のブランチとタグの一覧を表示する
import re
import subprocess

GITHUB_USER = "orgchem610"
GITHUB_REPO = "qchem_colab_app"
REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

result = subprocess.run(
    ["git", "ls-remote", "--heads", "--tags", REPO_URL], capture_output=True, text=True)
if result.returncode != 0:
    print("❌ 一覧を取得できませんでした。ネットワーク接続を確認してください。")
    print(result.stderr.strip()[-500:])
else:
    branches, tags = [], []
    for line in result.stdout.splitlines():
        ref = line.split("\t")[1]
        if ref.startswith("refs/heads/"):
            branches.append(ref[len("refs/heads/"):])
        elif ref.startswith("refs/tags/") and not ref.endswith("^{}"):
            tags.append(ref[len("refs/tags/"):])

    def natural_key(name):
        # v0.1.9 が v0.1.16 より後ろに並ばないよう、数字の部分は数として比較する
        return [int(part) if part.isdigit() else part for part in re.split(r"(\d+)", name)]

    print("【ブランチ】")
    for name in sorted(branches, key=natural_key):
        print("  ", name)
    print("【タグ(バージョン)】")
    for name in sorted(tags, key=natural_key):
        print("  ", name)
    print("\n上の名前のどれかを、次のセルの BRANCH 欄に選択(または入力)してください。")

In [ ]:
#@title 1-b. 選んだブランチ(またはタグ)のコードを取得する
import os
import re
import subprocess
import sys

BRANCH = "main"  #@param ["main", "develop"] {allow-input: true}

GITHUB_USER = "orgchem610"
GITHUB_REPO = "qchem_colab_app"
REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
PROJECT_DIR = f"/content/{GITHUB_REPO}"

BRANCH = BRANCH.strip()
if not re.fullmatch(r"[A-Za-z0-9._/-]+", BRANCH) or BRANCH.startswith("-"):
    raise ValueError(f"ブランチ名(タグ名)として使えない文字が含まれています: {BRANCH!r}")


def run_git(*args, cwd=None):
    """gitコマンドを実行し、(成功したか, 標準出力, 標準エラー) を返す。"""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    return r.returncode == 0, r.stdout.strip(), r.stderr.strip()


def fail(message, detail=""):
    print(f"❌ {message}")
    if detail:
        print(detail[-600:])
    print("ブランチ名・タグ名の綴りをご確認ください(一覧は「1-a」のセルで表示できます)。")
    raise RuntimeError(message)


previous_head = None
if not os.path.exists(PROJECT_DIR):
    print(f"{GITHUB_REPO} をクローンします(取得するブランチ/タグ: {BRANCH})...")
    ok, _, err = run_git("clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR)
    if not ok:
        fail("git clone に失敗しました。", err)
else:
    print(f"{PROJECT_DIR} は既に存在するため、{BRANCH} に切り替えて最新版を取得します...")
    _, previous_head, _ = run_git("rev-parse", "HEAD", cwd=PROJECT_DIR)
    ok, _, err = run_git("fetch", "--tags", "origin", cwd=PROJECT_DIR)
    if not ok:
        fail("GitHubからの情報取得(git fetch)に失敗しました。", err)
    ok, _, err = run_git("checkout", BRANCH, cwd=PROJECT_DIR)
    if not ok:
        fail(f"{BRANCH} への切り替え(git checkout)に失敗しました。", err)
    # ブランチなら最新の内容を取り込む。タグは内容が固定されているため、取り込みは不要。
    on_branch, _, _ = run_git("symbolic-ref", "-q", "HEAD", cwd=PROJECT_DIR)
    if on_branch:
        ok, _, err = run_git("pull", "--ff-only", cwd=PROJECT_DIR)
        if not ok:
            fail("最新版の取得(git pull)に失敗しました。", err)

_, head, _ = run_git("rev-parse", "HEAD", cwd=PROJECT_DIR)
_, current_branch, _ = run_git("branch", "--show-current", cwd=PROJECT_DIR)
_, last_commit, _ = run_git(
    "log", "-1", "--format=%h %s (%cd)", "--date=short", cwd=PROJECT_DIR)
_, described, _ = run_git("describe", "--tags", "--always", cwd=PROJECT_DIR)

print("\n=== 取得したコード ===")
if current_branch:
    print(f"  ブランチ        : {current_branch}")
else:
    print(f"  タグ(固定)    : {BRANCH}")
print(f"  最新コミット    : {last_commit}")
print(f"  git describe    : {described}")

if previous_head and previous_head != head and any(
        name in sys.modules for name in ("gui", "qcapp", "common")):
    print(
        "\n⚠ 取得したコードが前回から変わりましたが、Pythonは古いコードを読み込んだままです。\n"
        "  「ランタイム」→「セッションを再起動」を実行し、このセルから上から順に実行し直してください。")

In [ ]:
import sys
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)
print("sys.pathに追加しました:", PROJECT_DIR)

### 2. 依存ライブラリのインストール

`requirements-uma-lock.txt` が既にあればそちらを、なければ `requirements-uma.txt` を使って
インストールします(PySCF版の `requirements-lock.txt` とは別のファイルです)。
**初回実行後、`requirements-uma-lock.txt` が自動生成されます。**

パッケージを1つずつインストールし、失敗したものがあれば、**どのパッケージが・どんなエラーで失敗したか**を
表示します。UMAの本体(`fairchem-core`)は大きなライブラリのため、**初回は数分かかります。**

**初回のインストールでは、`fairchem-core` によってColab標準の `torch` が 2.11.0 から 2.13.0 に
更新されることを確認しています**(想定どおりの動作です)。更新されたライブラリが、すでにPythonに
読み込まれている場合だけ、**「ランタイム」→「セッションを再起動」** が必要になります(その旨を表示します)。
再起動した場合は、「1-b」のセルから実行し直してください(インストール済みのライブラリは残っています)。
インストール後は、「2-c」のセルで、torchとGPUが使えることを確認してください。

In [ ]:
#@title 2. 依存ライブラリをインストールする
import importlib.metadata as md
import os
import subprocess
import time

def _version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

# 「0. 現在の実行環境を確認する」を実行済みならその記録を、(セッション再起動後などで)
# 無ければ現在の版を、「インストール前の版」として扱う。
VERSIONS_BEFORE_INSTALL = globals().get("VERSIONS_BEFORE_INSTALL") or {
    "numpy": _version("numpy"), "torch": _version("torch"), "ipywidgets": _version("ipywidgets")}

req_lock = os.path.join(PROJECT_DIR, "requirements-uma-lock.txt")
req_plain = os.path.join(PROJECT_DIR, "requirements-uma.txt")
req_file = req_lock if os.path.exists(req_lock) else req_plain
print(f"インストールに使用するファイル: {req_file}")

with open(req_file) as f:
    lines = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]

failed = []
t0 = time.time()
for pkg in lines:
    print(f"--- installing: {pkg} ---")
    result = subprocess.run(["pip", "install", "-q", pkg], capture_output=True, text=True)
    if result.returncode != 0:
        failed.append(pkg)
        print(f"❌ {pkg} のインストールに失敗しました。エラー末尾:")
        print("\n".join(result.stderr.strip().splitlines()[-15:]))
    else:
        print(f"OK: {pkg}")
print(f"\nインストール所要時間: {time.time() - t0:.0f} 秒")

if failed:
    print("\n=== 以下のパッケージのインストールに失敗しました ===")
    for pkg in failed:
        print(" -", pkg)
    print(
        "\n上のエラー内容(特に最後の数行)を確認してください。\n"
        "ランタイムバージョンが 2026.07 でない場合は、torch/numpy/Pythonの版が合わず\n"
        "失敗することがあります(上の「0. 現在の実行環境を確認する」を再確認してください)。\n"
        "ColabReaction( https://github.com/BILAB/ColabReaction )では、fairchem-core の\n"
        "インストール前に numba==0.60.0 を指定しているため、numba関連のエラーの場合は\n"
        "参考にしてください。表示されたパッケージ名とエラー内容をそのままご連絡いただいても構いません。"
    )
else:
    print("\n全てのパッケージのインストールに成功しました。")

# torch / numpy の版が入れ替わっていないか確認する
import sys

changed = []
for pkg, before in VERSIONS_BEFORE_INSTALL.items():
    after = _version(pkg)
    if before != after:
        changed.append((pkg, before, after))
if changed:
    print("\nインストールにより、次のライブラリの版が変わりました:")
    for pkg, before, after in changed:
        print(f"  - {pkg}: {before} → {after}")
    if any(pkg == "ipywidgets" for pkg, _, _ in changed):
        print("\n(ipywidgetsの版が変わりました。8.x になった場合、セッションを再起動すると画面部品が"
              "表示されなくなることがあります。その場合は、Execution Sectionの「II-0」のセルを実行してください。)")
    already_loaded = [pkg for pkg, _, _ in changed if pkg in sys.modules and pkg != "ipywidgets"]
    if already_loaded:
        print(f"\n⚠ {', '.join(already_loaded)} は既にPythonに読み込まれているため、"
              "古い版のまま動いてしまいます。")
        print("「ランタイム」→「セッションを再起動」を実行してから、「1-b」のセルを含めて上から順に実行し直してください。")
    elif any(pkg != "ipywidgets" for pkg, _, _ in changed):
        print("(まだPythonに読み込まれていないため、セッションの再起動は不要です。次のセルへ進んでください。)")

In [ ]:
#@title 2-b. インストールされた版を requirements-uma-lock.txt に記録する
# Colabの pip freeze はプリインストール済みの大量のパッケージも含むため、
# requirements-uma.txt に明示的に書かれているパッケージだけに絞って書き出します
# (理由は README.md「requirements-lock.txtを軽量に保つ理由」と同じです)。
import re
import subprocess

freeze_result = subprocess.run(["pip", "freeze"], capture_output=True, text=True)
all_lines = freeze_result.stdout.splitlines()

with open(os.path.join(PROJECT_DIR, "requirements-uma.txt")) as f:
    wanted_names = set()
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            wanted_names.add(re.split(r"[<>=!~\[]", line)[0].strip().lower().replace("_", "-"))

def _pkg_name(line):
    return re.split(r"[<>=!~\[ @]", line)[0].strip().lower().replace("_", "-")

filtered_lines = [line for line in all_lines if _pkg_name(line) in wanted_names]

lock_path = os.path.join(PROJECT_DIR, "requirements-uma-lock.txt")
with open(lock_path, "w") as f:
    f.write("\n".join(sorted(filtered_lines)) + "\n")

print(f"requirements-uma-lock.txt を書き出しました({len(filtered_lines)}パッケージ分)。")
for line in sorted(filtered_lines):
    print(" -", line)
print("\n(この記録ファイルはGit管理の対象外にしてください。.gitignore に requirements-uma-lock.txt を追加します)")

In [ ]:
#@title 2-c. torchとGPUが使えることを確認する
# fairchem-core のインストールで torch が更新されるため(2.11.0 → 2.13.0)、
# 更新後のtorchがこのランタイムのGPUを使えることを確認します。
try:
    import torch
except ImportError as e:
    print("❌ torch を読み込めませんでした。「2. 依存ライブラリのインストール」を確認してください。")
    print(e)
else:
    print(f"torch : {torch.__version__}  (CUDA {torch.version.cuda})")
    if torch.cuda.is_available():
        print(f"GPU   : {torch.cuda.get_device_name(0)}")
        try:
            x = torch.randn(512, 512, device="cuda")
            float((x @ x).sum().item())
            print("\n✓ GPU上での計算に成功しました。")
        except Exception as e:
            print("\n❌ GPU上での計算に失敗しました:")
            print(e)
    else:
        print("\n⚠ torchからGPUを利用できません。このままではUMAの計算が非常に遅くなります。")
        print("  ・「ランタイム」→「ランタイムのタイプを変更」で、T4 GPUが選ばれているか確認してください。")
        print("  ・選ばれているのにこの表示になる場合は、更新後のtorchとColabのGPUドライバの組み合わせの")
        print("    問題の可能性があります。この表示をそのままご連絡ください。")

### 3. Hugging FaceへのログインとUMAへのアクセスの確認

インストール後に、冒頭で受け付けたトークンでHugging Faceにログインし、UMAにアクセスできることを確認します
(利用条件への同意・承認が済んでいない場合は、ここで案内を表示します。承認を待つ間も、他のセルは実行できます)。

In [ ]:
#@title 3. Hugging FaceへのログインとUMAへのアクセス確認
from uma_app import uma_engine

try:
    uma_engine.login_from_environment()
    uma_engine.check_model_access()
    print("✓ Hugging Faceにログインし、UMAへのアクセスを確認しました。")
except uma_engine.UmaSetupError as e:
    print("❌ UMAを使うための準備が完了していません:")
    print(e)
    print("\n原因を解消したら、このセルを実行し直してください。トークンを入れ直す場合は、"
          "冒頭のトークン入力セルで「FORCE_REENTER」にチェックを入れて実行してから、このセルを実行してください。")

### 4. 動作確認用サンプル構造の準備(任意)

初めて実行する方は、いきなりご自身の分子を試す前に、下のセルで作成される小さなサンプル
(エタノール, 9原子)で、パイプライン全体(構造最適化→振動数計算)が動くことを確認することをお勧めします。

In [ ]:
sample_xyz = """9
Ethanol (CH3CH2OH) - rough starting geometry for connectivity testing
C   0.000000   0.000000   0.000000
C   1.510000   0.000000   0.000000
O   1.970000   1.340000   0.000000
H  -0.363000   0.000000   1.028000
H  -0.363000  -0.890000  -0.514000
H  -0.363000   0.890000  -0.514000
H   2.020000  -0.360000   0.890000
H   2.020000  -0.360000  -0.890000
H   1.445000   2.144000   0.000000
"""
with open("/content/sample_ethanol.xyz", "w") as f:
    f.write(sample_xyz)
print("/content/sample_ethanol.xyz を作成しました。")
print("下の実行画面のドロップ領域(またはアップロードボタン)でこのファイルを選択して、まず動作確認することをお勧めします。")
print("(Colab左側のフォルダアイコンから /content 内のファイルとして参照できます)")

## II. Execution Section

以下のセルを実行すると操作画面が表示されます。

1. 上部の**UMAの利用条件の注意書き**を読み、確認のチェックを入れる
2. 「① 構造ファイルと計算条件」で、構造ファイル(`.xyz` / `.com` / `.gjf` / `.pdb`)を枠に
   ドラッグ＆ドロップ(または「構造ファイル」ボタンで選択)し、UMAモデル・電荷・スピン多重度を設定
3. 「② 実行」の **「計算を実行する」** ボタンを押す(初回はモデルのダウンロードに時間がかかります)
4. 「③ 可視化」でエネルギー推移・振動アニメーションを確認
   (エネルギーのグラフは、点をクリックするとその点の構造が下に表示されます)
5. 「④ ダウンロード」から最終構造(ColabReactionにそのまま渡せる.xyz)・トラジェクトリ・計算ログを取得

**条件を変えて再実行したい場合は、このセルをもう一度実行し直してください
(GUI部品が再生成され、状態がリセットされます)。**

**⚠ 操作画面が表示されず、右側に「コードスニペット」のパネル(`enable_custom_widget_manager` で絞り込まれた状態)が
開くだけの場合**は、ColabのGUI部品(ipywidgets)の版に関する問題です。下の「II-0」のセルを実行してから、
もう一度その下のセルを実行してください(ランタイムを新しく接続し直した場合は、通常は不要です)。

In [ ]:
#@title II-0. 画面部品(ウィジェット)を表示する準備
# Colab標準のウィジェット表示の仕組みは ipywidgets 7.x 向けです。ipywidgets 8.x が
# 読み込まれている場合は、「カスタムウィジェットマネージャー」を有効にしないと、
# 画面部品が表示されず「コードスニペット」のパネルが開くだけになります。
import ipywidgets

major = int(ipywidgets.__version__.split(".")[0])
print(f"ipywidgets: {ipywidgets.__version__}")
if major >= 8:
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
        print("✓ カスタムウィジェットマネージャーを有効にしました。次のセルを実行してください。")
    except ImportError:
        print("(Google Colab以外の環境のため、この設定は不要です。)")
else:
    print("Colab標準の仕組みで表示できるため、この設定は不要です。次のセルへ進んでください。")

In [ ]:
import gui_uma
from IPython.display import display

app = gui_uma.build_app()
display(app)